# Week 5a.1: Retrieval

The support agent answers policy questions out of `search_faq`, which is a dictionary with four entries in it. That worked because we wrote both the questions and the answers.

Real policies are documents. Husky Tech's runs to four sections covering returns, shipping, warranty and support hours, and the useful parts of it are sentences buried inside paragraphs. We cannot put a document in a prompt and we cannot write a dictionary entry for every question a customer might ask.

What we'll do instead is give the agent a way to look things up: split the document into pieces, store those pieces so they can be searched by meaning rather than by keyword, and retrieve the few that bear on the question. In other words, we'll add a RAG component to our agent.

- https://docs.langchain.com/oss/python/langchain/retrieval


### RAG Steps
- Ingest the knowledgebase
    - Load the docs
    - Chunk the docs
    - Embed the chunks
    - Store chunk embeddings in a vector store

- Retrieve the relevant chunks
    - Query the vector store with the user question
    - Get the matching chunks
    - Insert them into the prompt
    - Feed the augmented prompt into the model

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. Loading Documents

`husky_policy.md` is the written policy we'll use for the RAG example.

LangChain provides various document loader integrations:
- https://docs.langchain.com/oss/python/integrations/document_loaders


In [ ]:
policy_text = open("husky_policy.md").read()

print(f"{len(policy_text)} characters\n")
print(policy_text)

Notice section 1.3. It does not restate the return window; it says *"The 30-day window described in section 1.1 does not apply..."*. Policies are written like this, because a human reading the document can see section 1.1 a few lines above.

## 2. Splitting

It's not feasible to put the entire set of docs into the context window.

Instead, we'll split the document into chunks and retrieve only the ones that matter for the query, which we'll then feed into the model.

LangChain provides various integrations for text splitting:
- https://docs.langchain.com/oss/python/integrations/splitters

`RecursiveCharacterTextSplitter` tries to break on paragraph boundaries first, then lines, then words, so that chunks end at natural places rather than mid-sentence.
- **chunk_size** determines the number of characters
- **chunk_overlap** specifies the amount of overlapping characters between the chunks
- **split_text()** is used to split the document with the splitter.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = #TODO
chunks = #TODO

print(f"{len(chunks)} chunks\n")
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ({len(c)} chars)")
    print(c.strip()[:200], "\n")

Read chunk 0 and chunk 2 next to each other.

Chunk 0 contains the 30-day rule. Chunk 2 contains the exception to it, and refers to "section 1.1" to do so. They are now two separate pieces of text, and nothing connects them.

Hold on to that. It matters more than it looks, and we come back to it on Friday.


## 3. Embeddings

To search by meaning we need meaning as a number. An **embedding model** turns a piece of text into a vector, positioned so that texts about similar things land near each other.

LangChain provides various integrations for embedding models:
- https://docs.langchain.com/oss/python/integrations/embeddings


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

### Ollama: Qwen3 Embedding Model
Download the embedding model 
- https://ollama.com/library/qwen3-embedding

Run `ollama pull qwen3-embedding:0.6b` in the terminal

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")

### OpenRouter: Liquid LFM embedding

OpenRouter serves embeddings on an OpenAI-compatible endpoint, so the OpenAI client
works against it once you point it at their base URL. Add `OPENROUTER_API_KEY` to
your `.env`.

Note `check_embedding_ctx_length=False`. Left at its default, LangChain counts tokens
with OpenAI's tokenizer before sending, which this model rejects with a 400.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="liquid/lfm-2.5-embedding-350m:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    check_embedding_ctx_length=False,
)

### NVIDIA: Nemotron embedding

`nvidia/nemotron-3-embed-1b` runs on NVIDIA's hosted endpoint. Add `NVIDIA_API_KEY`
to your `.env`.

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

embeddings = NVIDIAEmbeddings(model="nvidia/nemotron-3-embed-1b")

In [ ]:
v = embeddings.embed_query("Can I return a damaged item?")
print(f"a vector of {len(v)} numbers")
print(v[:8], "...")

The numbers mean nothing on their own. 

What matters is distance, which will be our measure of similarity between two vectors.

Here is a simple cosine similarity function:

In [ ]:
import numpy as np

def similarity(x, y):
    x, y = np.array(x), np.array(y)
    return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))

In [ ]:
a = embeddings.embed_query("Can I return a damaged item?")
b = embeddings.embed_query("My headphones arrived broken, can I send them back?")
c = embeddings.embed_query("What time does support close on Friday?")

In [ ]:
len(a), len(b), len(c)

In [ ]:
print(f"damaged item   vs  arrived broken : {similarity(a, b):.3f}")
print(f"damaged item   vs  support hours  : {similarity(a, c):.3f}")

The two return questions score higher than the return question against the support-hours question, and they share no important words. That is the whole idea: the match is on meaning, not on vocabulary.

## 4. Embedding the documents

So far we have embedded three sentences we typed by hand. The policy itself has not been embedded at all.

We'll now embed the policy document using the embedding model.
 - `embed_query` takes one string, which is what a question is. 
 - `embed_documents` takes the whole list, which is what a corpus is. 

**The document and queries need to be embedded using the same model, because vectors from two different models cannot be compared.**

In [ ]:
chunks

In [ ]:
# TODO: get the vectors for the chunks

print(f"{len(chunks)} chunks -> {len(vectors)} vectors of {len(vectors[0])} numbers each")

In [ ]:
type(vectors)

In [ ]:
len(vectors)

In [ ]:
vectors[0][:5]

That is the expensive step, and it runs **once**, when the document changes. Every question after it only embeds the question.


## 5. Storing the documents
A vector store holds the chunks and their vectors and answers "which of these is closest?". 

`InMemoryVectorStore` keeps everything in a Python list. It disappears when the kernel stops, which is fine for a notebook but not acceptable for an application.

`from_texts` does the embedding above for you, which is why you will not usually see `embed_documents` written out.
 - `from_text(chunks, embeddings)`



In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

# TODO create the vector store

In [ ]:
type(store)

### store.store.items()
- A database containing the id, the embedding vector, and the content for the chunk.

In [ ]:
for item in store.store.items():
    print(item[1]["id"])
    print(item[1]["vector"][:3])
    print(item[1]["text"][:50])
    print()

### Querying the vector store

`similarity_search()` allows you to query the vector store with a given string.
- First, the query is embedded using the embeddings model specified in from_texts()
- `k` specifies the number of chunks to return

In [ ]:
hits = store.similarity_search("how long does shipping take?", k=2)
for h in hits:
    print("-", " ".join(h.page_content.split())[:110], "\n")

In [ ]:
hits

In [ ]:
hits[0].page_content.strip()

## 6. Retrieval as a tool

The agent needs this as a tool it can call. Note what the tool returns: the text of the chunks, not an answer. The retrieval step finds evidence, and the model does the reasoning.

In [ ]:
from langchain.tools import tool

@tool
def search_policy(question: str) -> str:
    """Search the Husky Tech written policy and return the passages most relevant
    to a question. Use this for any question about returns, shipping, warranty or
    support hours. Returns policy text, not an answer: read it and decide."""
    
    #TODO: get the hits from the vector store for the question with k=2
    
    if not hits:
        return "Nothing in the policy matched that question."
    return "\n\n---\n\n".join(h.page_content.strip() for h in hits)

In [ ]:
print(search_policy.invoke({"question": "do you ship to Canada?"}))

## 7. The agent with retrieval

Same graph as W04b. The only change is the tool list: `search_policy` replaces `search_faq`, and everything else stays exactly as it was.

In [ ]:
from support_tools import look_up_order, check_return_eligibility, escalate_to_human, SYSTEM_PROMPT
from langchain.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from IPython.display import Image, display


sys_msg = SystemMessage(content=SYSTEM_PROMPT)
TOOLS = [look_up_order, check_return_eligibility, search_policy, escalate_to_human]
model_with_tools = model.bind_tools(TOOLS)

def assistant(state: MessagesState) -> dict:
    return {"messages": [model_with_tools.invoke([sys_msg] + state["messages"])]}

#TODO: build the graph
#TPDP: create the support_agent

display(Image(support_agent.get_graph().draw_mermaid_png()))

It answers policy questions now, from a document nobody wrote a dictionary entry for:

In [ ]:
config = {"configurable": {"thread_id": "customer-1"}}

for q in ["How long does standard shipping take?",
          "Can I return opened software?",
          "What does the warranty cover?"]:
    result = support_agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Q: {q}")
    print(f"A: {result['messages'][-1].text.strip()[:200]}\n")

## 8. Using a real vector store: Pinecone

`InMemoryVectorStore` has done everything we asked of it, and it has one property
that rules it out for anything real. It is a Python dictionary. When the kernel
stops the index is gone, rebuilding it means paying to embed every chunk again, and
nothing outside this process can read it. The indexing job and the agent are stuck
living in the same notebook.

A managed vector database addresses these limitations. The index sits on a server, it persists,
and anything holding the API key can query it.

We'll use Pinecone. The free tier is more than enough for this course.
- https://docs.langchain.com/oss/python/integrations/vectorstores/pinecone

**Setup**

1. `uv pip install langchain-pinecone`
2. Make an account at [pinecone.io](https://www.pinecone.io) and create an API key
3. Add `PINECONE_API_KEY=...` to your `.env`, next to the keys you already have

In [ ]:
load_dotenv()
assert os.getenv("PINECONE_API_KEY"), "No PINECONE_API_KEY found in .env"
print("Pinecone key loaded")

### The index has a fixed width

An index is created with a number of dimensions, and every vector you put in it has
to be exactly that wide. That number comes from the embedding model you picked in
section 3. Gemini returns 3072 numbers and `qwen3-embedding:0.6b` returns 1024, so
the two are not interchangeable.

Rather than hardcode it, ask the model. This is the same fact from section 4 in a
place where it has consequences: the width belongs to the model, not to the text.

In [ ]:
dimension = len(embeddings.embed_query("a probe to measure the width"))
index_name = f"husky-policy-{dimension}"

print(f"{index_name}  ->  {dimension} dimensions")

### Accessing the index


In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "rag-demo"
index = pc.Index(index_name)

In [ ]:
index.describe_index_stats()

### Putting the chunks in
The index exists and is empty. 

`metric="cosine"` has to match how you want nearness measured, and it is what our `similarity` function computed by hand in section 3. 

`dimension` has to match the model.

`add_texts` embeds each chunk and uploads it with its text attached, which is the indexing half of the RAG pipeline.
- This runs **once per document change**, not once per question. Writes take a moment
to become visible, so if the count below reads zero, run the stats cell again.

In [ ]:
from langchain_pinecone import PineconeVectorStore

pinecone_store = PineconeVectorStore(index=index, embedding=embeddings)
pinecone_store.add_texts(chunks)

print(f"uploaded {len(chunks)} chunks")

In [ ]:
index.describe_index_stats()

### Searching the index

This is the same method call we made against `InMemoryVectorStore`.

In [ ]:
hits = pinecone_store.similarity_search("how long does shipping take?", k=2)

for h in hits:
    print("-", " ".join(h.page_content.split())[:110], "\n")

In [ ]:
hits[0]

Note that `similarity_search(query, k)` is the same call with the same arguments returning the same objects, because both classes implement LangChain's `VectorStore` interface. 

The store behind it changed completely and the code asking the question did not change at all.

So the tool only needs one line altered.

In [ ]:
@tool
def search_policy(question: str) -> str:
    """Search the Husky Tech written policy and return the passages most relevant
    to a question. Use this for any question about returns, shipping, warranty or
    support hours. Returns policy text, not an answer: read it and decide."""
    hits = pinecone_store.similarity_search(question, k=2)      # <- the only change
    if not hits:
        return "Nothing in the policy matched that question."
    return "\n\n---\n\n".join(h.page_content.strip() for h in hits)


TOOLS = [look_up_order, check_return_eligibility, search_policy, escalate_to_human]
model_with_tools = model.bind_tools(TOOLS)

builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

support_agent = builder.compile(checkpointer=InMemorySaver())

display(Image(support_agent.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "customer-2"}}

result = support_agent.invoke(
    {"messages": [HumanMessage(content="Can I return opened software?")]}, config)

print(result["messages"][-1].text)

### What you have now that you did not have before

**It persists.** Restart the kernel and the index is still there with the chunks in
it. We don't have to embed the docs again.

**Anything can read it.** A deployed app, a scheduled job and this notebook can all
query the same index. Indexing and answering no longer have to be the same program.

**It filters.** Each record can carry metadata, and you can restrict a search to
records matching it. This is how a real system keeps one customer from retrieving
another customer's documents.

**It scales differently.** `InMemoryVectorStore` compares your question against every
vector it holds, which is exact and perfectly fine for six chunks. Pinecone uses an
approximate index, which gives up a little accuracy to stay fast at millions of
vectors.

## ICA

1. Rebuild the store with `chunk_size=1000`, then again with `chunk_size=200`. Print
   the chunks each time and ask the same three questions. Where do the retrieved
   passages start carrying text you did not ask about, and where do they start losing
   the sentence they needed?

2. Set `chunk_overlap=100` at the original size of 400. Compare the chunk boundaries
   against the run with no overlap. What does the overlap buy you, and what does it
   cost you inside a top-2 result?

3. `search_policy` asks for the 2 closest chunks. Try 1, 3 and 6 on "what are your
   support hours" and read what the model has to work through each time. Then say what
   you would set `k` to if you had to pick a single number that served every question
   a customer might ask.

4. Switch the embedding model in section 3, from Gemini to Ollama or back, and rerun
   section 7. You will find you need a new index. Explain why the existing one cannot
   be reused.

5. Ask three questions of your own that the policy does answer. Find one where the
   retrieved chunks are not the ones you would have chosen yourself, and work out
   whether the cause is the chunking, the embedding model, or the way the question was
   worded.

In [ ]:
# your experiments here